> Copyright 2024 DeepMind Technologies Limited.
>
> Licensed under the Apache License, Version 2.0 (the "License");
> you may not use this file except in compliance with the License.
> You may obtain a copy of the License at
>
>      http://www.apache.org/licenses/LICENSE-2.0
>
> Unless required by applicable law or agreed to in writing, software
> distributed under the License is distributed on an "AS-IS" BASIS,
> WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
> See the License for the specific language governing permissions and
> limitations under the License.

# GenCast Mini Demo

This notebook demonstrates running `GenCast 1p0deg Mini <2019`.

`GenCast 1p0deg Mini <2019` is a GenCast model at 1deg resolution, with 13 pressure levels and a 4 times refined icosahedral mesh. It is trained on ERA5 data from 1979 to 2018, and can be causally evaluated on 2019 and later years.

While other GenCast models are [available](https://github.com/google-deepmind/graphcast/blob/main/README.md), this model has the smallest memory footprint of those provided and is the only one runnable with the freely provided TPUv2-8 configuration in Colab. You can select this configuration in `Runtime>Change Runtime Type`.

**N.B.** The performance of `GenCast 1p0deg Mini <2019` is reasonable but is not representative of the performance of the other GenCast models described in the [README](https://github.com/google-deepmind/graphcast/blob/main/README.md).

To run the other models using Google Cloud Compute, refer to [gencast_demo_cloud_vm.ipynb](https://colab.research.google.com/github/deepmind/graphcast/blob/master/gencast_demo_cloud_vm.ipynb).

## Running this on Kaggle

This copy has been adapted to run as a **Kaggle Notebook** instead of Colab.

Before running top to bottom:

1. **Upload your data as a Kaggle Dataset.** Zip up your local `gencast` folder
   (the one containing `dataset/`, `params/`, and `stats/` subfolders — the
   files you downloaded from the GCS bucket) and create a new Kaggle Dataset
   from it (kaggle.com -> Create -> New Dataset, or `kaggle datasets create`
   via the CLI). Attach that dataset to this notebook (Add Input).
2. **Set the accelerator.** Notebook settings -> Accelerator -> choose a GPU
   (T4 x2 or P100). This notebook is written for GPU, not TPU.
3. **Turn Internet on.** Notebook settings -> Internet -> On, so the install
   cell below can pip-install the model code.
4. Update `KAGGLE_DATASET_SLUG` in the next code cell to match the dataset
   slug you created (visible in the dataset's URL / in the Input panel on the
   right once attached), and update the filenames if yours differ.


# Installation and Initialization

In [ ]:
# @title Install the model code (pinned commit, matches this notebook's imports)

# The upstream repo (google-deepmind/weathernext, formerly google-deepmind/graphcast)
# was restructured after this commit: the `graphcast` package was split into
# `weathernext/utils`, `weathernext/weathernext1_gen` (GenCast), etc. Pinning to
# this commit keeps the `from graphcast import ...` imports below working as-is.
%pip install -q "git+https://github.com/google-deepmind/weathernext.git@08cf73625c9d12bd9aaa038868bcb2fe488f2a22"


In [ ]:
# @title Install a GPU-enabled JAX build (skip this cell if using a TPU accelerator instead)

%pip install -q -U "jax[cuda12]"

import jax
print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


In [ ]:
# @title Imports

import dataclasses
import datetime
import math
from typing import Optional
import haiku as hk
from IPython.display import HTML
from IPython import display
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray

from graphcast import rollout
from graphcast import xarray_jax
from graphcast import normalization
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import xarray_tree
from graphcast import gencast
from graphcast import denoiser
from graphcast import nan_cleaning


In [7]:
# @title Plotting functions

def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
  plt.close(figure.number)
  return HTML(ani.to_jshtml())


# Load the Data and initialize the model

In [ ]:
# @title Locate the GenCast files on Kaggle

from pathlib import Path

KAGGLE_DATASET_SLUG = "your-username/gencast-mini-data"  # @param {type:"string"}
DATA_DIR = Path("/kaggle/input") / KAGGLE_DATASET_SLUG.split("/")[-1]

DATA_PATH = DATA_DIR / "dataset" / "source-era5_date-2019-03-29_res-1.0_levels-13_steps-30.nc"
MODEL_PATH = DATA_DIR / "params" / "GenCast_1p0deg_Mini_lt2019.npz"
STATS_DIR = DATA_DIR / "stats"

required_files = [
    DATA_PATH,
    MODEL_PATH,
    STATS_DIR / "diffs_stddev_by_level.nc",
    STATS_DIR / "mean_by_level.nc",
    STATS_DIR / "stddev_by_level.nc",
    STATS_DIR / "min_by_level.nc",
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing files (check KAGGLE_DATASET_SLUG and that the dataset is "
        "attached to this notebook, and that the folder layout inside the "
        "dataset matches dataset/, params/, stats/):\n"
        + "\n".join(map(str, missing_files))
    )

print("Found all required GenCast files under:", DATA_DIR)


In [ ]:
# @title Load the model checkpoint

with open(MODEL_PATH, "rb") as f:
    ckpt = checkpoint.load(f, gencast.CheckPoint)

params = ckpt.params
state = {}

task_config = ckpt.task_config
sampler_config = ckpt.sampler_config
noise_config = ckpt.noise_config
noise_encoder_config = ckpt.noise_encoder_config
denoiser_architecture_config = ckpt.denoiser_architecture_config

print("Model description:\n", ckpt.description)
print("Model license:\n", ckpt.license)


In [ ]:
# @title Load the example ERA5 data

example_batch = xarray.open_dataset(DATA_PATH).compute()
assert example_batch.dims["time"] >= 3  # 2 for input, >=1 for targets

print("Dataset dimensions:", dict(example_batch.sizes))
example_batch


In [ ]:
# @title Plot example data (sanity check)

PLOT_EXAMPLE_VARIABLE = "2m_temperature"  # change to any name in example_batch.data_vars
PLOT_EXAMPLE_LEVEL = 500
PLOT_EXAMPLE_ROBUST = True

plot_size = 7
data = {
    " ": scale(
        select(example_batch, PLOT_EXAMPLE_VARIABLE, PLOT_EXAMPLE_LEVEL, example_batch.dims["time"]),
        robust=PLOT_EXAMPLE_ROBUST),
}
fig_title = PLOT_EXAMPLE_VARIABLE
if "level" in example_batch[PLOT_EXAMPLE_VARIABLE].coords:
    fig_title += f" at {PLOT_EXAMPLE_LEVEL} hPa"

plot_data(data, fig_title, plot_size, PLOT_EXAMPLE_ROBUST)


In [ ]:
# @title Extract training and eval data

train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("12h", "12h"), # Only 1AR training.
    **dataclasses.asdict(task_config))

eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("12h", f"{(example_batch.dims['time']-2)*12}h"), # All but 2 input frames.
    **dataclasses.asdict(task_config))

print("All Examples:  ", example_batch.dims.mapping)
print("Train Inputs:  ", train_inputs.dims.mapping)
print("Train Targets: ", train_targets.dims.mapping)
print("Train Forcings:", train_forcings.dims.mapping)
print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)


In [ ]:
# @title Load normalization data

diffs_stddev_by_level = xarray.load_dataset(STATS_DIR / "diffs_stddev_by_level.nc").compute()
mean_by_level = xarray.load_dataset(STATS_DIR / "mean_by_level.nc").compute()
stddev_by_level = xarray.load_dataset(STATS_DIR / "stddev_by_level.nc").compute()
min_by_level = xarray.load_dataset(STATS_DIR / "min_by_level.nc").compute()


In [ ]:
# @title Build jitted functions, and possibly initialize random weights


def construct_wrapped_gencast():
  """Constructs and wraps the GenCast Predictor."""
  predictor = gencast.GenCast(
      sampler_config=sampler_config,
      task_config=task_config,
      denoiser_architecture_config=denoiser_architecture_config,
      noise_config=noise_config,
      noise_encoder_config=noise_encoder_config,
  )

  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level,
  )

  predictor = nan_cleaning.NaNCleaner(
      predictor=predictor,
      reintroduce_nans=True,
      fill_value=min_by_level,
      var_to_clean='sea_surface_temperature',
  )

  return predictor


@hk.transform_with_state
def run_forward(inputs, targets_template, forcings):
  predictor = construct_wrapped_gencast()
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


@hk.transform_with_state
def loss_fn(inputs, targets, forcings):
  predictor = construct_wrapped_gencast()
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics),
  )


def grads_fn(params, state, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), i, t, f
    )
    return loss, (diagnostics, next_state)

  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True
  )(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads


if params is None:
  init_jitted = jax.jit(loss_fn.init)
  params, state = init_jitted(
      rng=jax.random.PRNGKey(0),
      inputs=train_inputs,
      targets=train_targets,
      forcings=train_forcings,
  )


loss_fn_jitted = jax.jit(
    lambda rng, i, t, f: loss_fn.apply(params, state, rng, i, t, f)[0]
)
grads_fn_jitted = jax.jit(grads_fn)
run_forward_jitted = jax.jit(
    lambda rng, i, t, f: run_forward.apply(params, state, rng, i, t, f)[0]
)
# We also produce a pmapped version for running in parallel.
run_forward_pmap = xarray_jax.pmap(run_forward_jitted, dim="sample")

# Run the model

The `chunked_prediction_generator_multiple_runs` iterates over forecast steps, where the 1 step forecast is jitted and samples are pmapped across the chips.
This allows us to make efficient use of all devices and parallelise generating an ensemble across them. We then combine the chunks at the end to form our final forecast.

Note that the `Autoregressive rollout` cell will take longer than the standard inference time to run when executed for the first time, as this will include code compilation time. This cost does not increase with the number of devices, it is a fixed-cost one time operation whose result can be reused across any number of devices.

In [ ]:
# The number of ensemble members should be a multiple of the number of devices.
print(f"Number of local devices {len(jax.local_devices())}")

In [ ]:
# @title Autoregressive rollout (loop in python)

print("Inputs:  ", eval_inputs.dims.mapping)
print("Targets: ", eval_targets.dims.mapping)
print("Forcings:", eval_forcings.dims.mapping)

num_ensemble_members = 4  # @param int  (start small on a single Kaggle GPU; raise once this runs cleanly)
rng = jax.random.PRNGKey(0)
# We fold-in the ensemble member, this way the first N members should always
# match across different runs which use take the same inputs, regardless of
# total ensemble size.
rngs = np.stack(
    [jax.random.fold_in(rng, i) for i in range(num_ensemble_members)], axis=0)

chunks = []
for chunk in rollout.chunked_prediction_generator_multiple_runs(
    # Use pmapped version to parallelise across devices.
    predictor_fn=run_forward_pmap,
    rngs=rngs,
    inputs=eval_inputs,
    targets_template=eval_targets * np.nan,
    forcings=eval_forcings,
    num_steps_per_chunk = 1,
    num_samples = num_ensemble_members,
    pmap_devices=jax.local_devices()
    ):
    chunks.append(chunk)
predictions = xarray.combine_by_coords(chunks)

In [ ]:
# @title Choose predictions to plot

PLOT_PRED_VARIABLE = "2m_temperature"  # change to any name in predictions.data_vars
PLOT_PRED_LEVEL = 500
PLOT_PRED_ROBUST = True
PLOT_PRED_MAX_STEPS = predictions.dims["time"]
PLOT_PRED_SAMPLES = num_ensemble_members


In [ ]:
# @title Plot prediction samples and diffs

plot_size = 5
plot_max_steps = min(predictions.dims["time"], PLOT_PRED_MAX_STEPS)

fig_title = PLOT_PRED_VARIABLE
if "level" in predictions[PLOT_PRED_VARIABLE].coords:
  fig_title += f" at {PLOT_PRED_LEVEL} hPa"

for sample_idx in range(PLOT_PRED_SAMPLES):
  data = {
      "Targets": scale(select(eval_targets, PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps), robust=PLOT_PRED_ROBUST),
      "Predictions": scale(select(predictions.isel(sample=sample_idx), PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps), robust=PLOT_PRED_ROBUST),
      "Diff": scale((select(eval_targets, PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps) -
                          select(predictions.isel(sample=sample_idx), PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps)),
                        robust=PLOT_PRED_ROBUST, center=0),
  }
  display.display(plot_data(data, fig_title + f", Sample {sample_idx}", plot_size, PLOT_PRED_ROBUST))


In [ ]:
# @title Plot ensemble mean and CRPS

def crps(targets, predictions, bias_corrected = True):
  if predictions.sizes.get("sample", 1) < 2:
    raise ValueError(
        "predictions must have dim 'sample' with size at least 2.")
  sum_dims = ["sample", "sample2"]
  preds2 = predictions.rename({"sample": "sample2"})
  num_samps = predictions.sizes["sample"]
  num_samps2 = (num_samps - 1) if bias_corrected else num_samps
  mean_abs_diff = np.abs(
      predictions - preds2).sum(
          dim=sum_dims, skipna=False) / (num_samps * num_samps2)
  mean_abs_err = np.abs(targets - predictions).sum(dim="sample", skipna=False) / num_samps
  return mean_abs_err - 0.5 * mean_abs_diff


plot_size = 5
plot_max_steps = min(predictions.dims["time"], PLOT_PRED_MAX_STEPS)

fig_title = PLOT_PRED_VARIABLE
if "level" in predictions[PLOT_PRED_VARIABLE].coords:
  fig_title += f" at {PLOT_PRED_LEVEL} hPa"

data = {
    "Targets": scale(select(eval_targets, PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps), robust=PLOT_PRED_ROBUST),
    "Ensemble Mean": scale(select(predictions.mean(dim=["sample"]), PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps), robust=PLOT_PRED_ROBUST),
    "Ensemble CRPS": scale(crps((select(eval_targets, PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps)),
                        select(predictions, PLOT_PRED_VARIABLE, PLOT_PRED_LEVEL, plot_max_steps)),
                      robust=PLOT_PRED_ROBUST, center=0),
}
display.display(plot_data(data, fig_title, plot_size, PLOT_PRED_ROBUST))


# Train the model

The following operations requires larger amounts of memory than running inference.

The first time executing the cell takes more time, as it includes the time to jit the function.

In [ ]:
# @title Loss computation
loss, diagnostics = loss_fn_jitted(
    jax.random.PRNGKey(0),
    train_inputs,
    train_targets,
    train_forcings)
print("Loss:", float(loss))

In [ ]:
# @title Gradient computation
loss, diagnostics, next_state, grads = grads_fn_jitted(
    params=params,
    state=state,
    inputs=train_inputs,
    targets=train_targets,
    forcings=train_forcings)
mean_grad = np.mean(jax.tree_util.tree_flatten(jax.tree_util.tree_map(lambda x: np.abs(x).mean(), grads))[0])
print(f"Loss: {loss:.4f}, Mean |grad|: {mean_grad:.6f}")